# Colab 14 — High-similarity sharpening with band-weighted MSE

## Pivot in framing

Colab10–13 chased aggregate function approximation across the full `[0, 1]` range. Colab13 disentangled the colab12 natural-pair collapse into a specific finding: the alphabet-entropy floor at `normLev ≈ 0.28` caps the achievable training range for equal-length AA pairs, and the encoder emits a flat `~0.44` default below that.

For colab14, the thesis claim narrows to what is **biologically meaningful**: we care about distinguishing *high-similarity* pairs (likely homologous, same fold/function) from *mid-similarity* (twilight-zone remote homology) from *far* pairs (below detection — random AA pairs share ~28% positions by chance anyway). Resolution below the 0.28 floor is biologically uninteresting.

## Bands

- **High:** `normLev ≥ 0.70` — clearly homologous
- **Mid:** `0.30 ≤ normLev < 0.70` — twilight zone
- **Far:** `normLev < 0.30` — below the alphabet-entropy floor; biology says random

## What changes vs. colab13

- **Loss:** plain MSE → band-weighted MSE with `w_high=4, w_mid=2, w_far=0.5`. Coverage is unchanged; the loss now spends most of its budget on the bands we care about.
- **Eval:** the band-blind transfer ladder is replaced with a sharper diagnostic suite — per-band Pearson r, binary AUROC for is-high-sim, PR curve, 3×3 confusion matrix, prediction histograms per band, identity-pair test, top-k retrieval against true Levenshtein neighbors, embedding-distance histograms, and PCA / t-SNE visualizations of the embedding geometry.

Architecture, sampler, vocab, length filter — all identical to colab13.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
import os
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz',
          'cath_s20_pairs_sample.csv.gz', 'cath_s20_pairs_high.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch torchvision rapidfuzz scikit-learn --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import roc_auc_score, precision_recall_curve, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from rapidfuzz.distance import Levenshtein as RFLevenshtein

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Constants — including band thresholds and weights

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
SS_ALPHABET = 'HLS'
VOCAB_SIZE = len(AA_ALPHABET) + 1
PAD_IDX = len(AA_ALPHABET)
AA_SET = set(AA_ALPHABET)
SS_SET = set(SS_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

MIN_LEN = 50
MAX_LEN_SEQ = 200
MAX_LEN = 200

N_TRAIN = 30000
N_HELD = 5000
NUM_EPOCHS = 30
BATCH_SIZE = 128

# Band thresholds + weights for the new banded MSE loss
BAND_LOW  = 0.30    # below this = far (don't care)
BAND_HIGH = 0.70    # at/above this = high (care most)
W_FAR  = 0.5
W_MID  = 2.0
W_HIGH = 4.0

# Top-k retrieval setup
TOPK_N_QUERIES   = 50
TOPK_N_CANDIDATES = 500
TOPK_VALUES = (1, 5, 10)

print(f'Bands: far<{BAND_LOW}, mid in [{BAND_LOW},{BAND_HIGH}), high>={BAND_HIGH}')
print(f'Weights: far={W_FAR}, mid={W_MID}, high={W_HIGH}')

## 3. Helpers — Levenshtein, perturb, encode, band labeling

In [ ]:
def levenshtein(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n]

def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - levenshtein(a, b) / L

def fast_norm_lev(a, b):
    """rapidfuzz-backed normLev (C implementation, ~100x faster)."""
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - RFLevenshtein.distance(a, b) / L

def is_standard_aa(seq): return all(c in AA_SET for c in seq)
def is_standard_ss(seq): return all(c in SS_SET for c in seq)

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    idx = [CHAR_TO_IDX[c] for c in seq][:max_len]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

def perturb(seq, k, alphabet, max_len=MAX_LEN):
    s = list(seq); abc = list(alphabet)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= max_len: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s))
            choices = [c for c in abc if c != s[i]]
            s[i] = rng.choice(choices)
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1)
            s.insert(i, rng.choice(abc))
        elif op == 'del':
            i = rng.integers(0, len(s))
            del s[i]
    return ''.join(s)

def random_aa_seq(min_len=MIN_LEN, max_len=MAX_LEN_SEQ):
    length = int(rng.integers(min_len, max_len + 1))
    return ''.join(rng.choice(list(AA_ALPHABET), size=length))

def band_label(x):
    """Return 'far', 'mid', or 'high' for a similarity value (numpy/python scalar)."""
    if x < BAND_LOW:  return 'far'
    if x < BAND_HIGH: return 'mid'
    return 'high'

def band_labels_arr(xs):
    xs = np.asarray(xs)
    out = np.full(xs.shape, 'mid', dtype=object)
    out[xs < BAND_LOW]  = 'far'
    out[xs >= BAND_HIGH] = 'high'
    return out

## 4. Build training pairs — single-procedure, target-uniform sampling

Unchanged from colab13. Sample `t ∼ Uniform[0, 1]`, pick `k = round((1−t)·L)`, apply `perturb`, and use the actual computed `normLev` as the label. All training pairs come from the same procedure (no procedural axis the encoder can shortcut on).

The achievable label range is `[~0.28, 1.0]` because of the alphabet-entropy floor. That is intentional: below 0.28 is the "far" band where we explicitly don't care about resolution — the band-weighted loss will only need to teach the model "this is far" rather than "this is exactly 0.13".

In [ ]:
def make_targeted_perturbation_pairs(seed_fn, alphabet, n_pairs):
    pairs = []
    attempts = 0
    max_attempts = n_pairs * 4
    while len(pairs) < n_pairs and attempts < max_attempts:
        attempts += 1
        seed = seed_fn()
        if seed is None or len(seed) < MIN_LEN: continue
        L = len(seed)
        t = float(rng.uniform(0.0, 1.0))
        k = max(0, int(round((1.0 - t) * L)))
        other = perturb(seed, k, alphabet)
        if 1 <= len(other) <= MAX_LEN:
            label = norm_lev(seed, other)
            pairs.append((seed, other, label))
    return pairs

print('Building training pairs (target-uniform random-AA perturbations)…')
train_pairs = make_targeted_perturbation_pairs(
    seed_fn=random_aa_seq, alphabet=AA_ALPHABET, n_pairs=N_TRAIN)
print(f'  got {len(train_pairs)} training pairs')

labels = np.array([p[2] for p in train_pairs])
band_counts = {b: int((band_labels_arr(labels) == b).sum()) for b in ('far','mid','high')}
print(f'  label range:    [{labels.min():.3f}, {labels.max():.3f}]')
print(f'  band counts:    {band_counts}')

plt.figure(figsize=(8, 3))
plt.hist(labels, bins=40, edgecolor='k', alpha=0.75)
plt.axvline(BAND_LOW,  color='r', ls='--', label=f'far/mid threshold ({BAND_LOW})')
plt.axvline(BAND_HIGH, color='g', ls='--', label=f'mid/high threshold ({BAND_HIGH})')
plt.xlabel('Label (normLev similarity)'); plt.ylabel('Count')
plt.title('Training-pair label distribution with band thresholds')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 5. Dataset / Architecture (unchanged from colab13)

In [ ]:
class PairDataset(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        a, b, label = self.pairs[i]
        return encode_pad(a), encode_pad(b), torch.tensor(label, dtype=torch.float32)

train_dl = DataLoader(PairDataset(train_pairs), batch_size=BATCH_SIZE, shuffle=True)

class SiameseEncoder(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, hidden1=32, hidden2=64,
                 max_len=MAX_LEN, out_dim=128, pad_idx=PAD_IDX):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        self.fc = nn.Linear(hidden2 * max_len, out_dim)
    def forward(self, x):
        mask = (x != self.pad_idx).float()
        x = self.embedding(x).permute(0, 2, 1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x * mask.unsqueeze(1)
        x = x.flatten(1)
        x = self.fc(x)
        return F.normalize(x, p=2, dim=1)

class SiameseNetwork(nn.Module):
    def __init__(self, **kw):
        super().__init__(); self.encoder = SiameseEncoder(**kw)
    def forward(self, a, b):
        ea, eb = self.encoder(a), self.encoder(b)
        return 1.0 - torch.linalg.vector_norm(ea - eb, ord=2, dim=1) / 2.0
    def encode(self, x): return self.encoder(x)

model = SiameseNetwork().to(device)
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

## 6. Training — band-weighted MSE

For each pair, the squared error term is multiplied by a per-sample weight determined by the *true* label's band:

- label `< 0.30`: weight `0.5` (far — we don't care about resolution)
- `0.30 ≤ label < 0.70`: weight `2.0` (mid — twilight zone)
- label `≥ 0.70`: weight `4.0` (high — biologically meaningful)

The model is forced to spend its capacity where it matters most. Plain MSE is recovered by setting all weights to 1.0.

In [ ]:
def banded_mse_loss(pred, label):
    weights = torch.ones_like(label)
    weights[label < BAND_LOW] = W_FAR
    weights[(label >= BAND_LOW) & (label < BAND_HIGH)] = W_MID
    weights[label >= BAND_HIGH] = W_HIGH
    return (((pred - label) ** 2) * weights).mean()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []
model.train()
for epoch in range(1, NUM_EPOCHS + 1):
    epoch_loss = 0.0; nb = 0
    for a, b, label in train_dl:
        a = a.to(device); b = b.to(device); label = label.to(device)
        pred = model(a, b)
        loss = banded_mse_loss(pred, label)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item(); nb += 1
    avg = epoch_loss / nb
    losses.append(avg)
    if epoch % 2 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS} — banded-MSE: {avg:.5f}')

plt.figure(figsize=(8, 4))
plt.plot(losses); plt.xlabel('Epoch'); plt.ylabel('Band-weighted MSE')
plt.title('Training loss — band-weighted MSE')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f'Final loss: {losses[-1]:.5f}')

## 7. Held-out eval sets

Three perturbation-based held-outs (random AA in-domain, real CATH AA, real CATH SS cross-rep) plus two natural-pair eval sets from the test30 slice of `cath_s20_pairs_*.csv.gz`.

In [ ]:
test_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_test30.csv.gz'),
                     compression='gzip').dropna(subset=['aa_seq', 'ss_seq'])
test_df['aa_seq'] = test_df['aa_seq'].astype(str)
test_df['ss_seq'] = test_df['ss_seq'].astype(str)
test_df = test_df[test_df['aa_seq'].str.len().between(MIN_LEN, MAX_LEN_SEQ)]
test_df = test_df[test_df['aa_seq'].apply(is_standard_aa)]
test_df = test_df[test_df['ss_seq'].apply(is_standard_ss)]
test_df = test_df[test_df['aa_seq'].str.len() == test_df['ss_seq'].str.len()]
test_df = test_df.reset_index(drop=True)
print(f'test30 proteins kept: {len(test_df)}')

test_aa_seqs = test_df['aa_seq'].tolist()
test_ss_seqs = test_df['ss_seq'].tolist()

print('\nBuilding held-out 1: random-AA targeted perturbations (in-domain)…')
held_random = make_targeted_perturbation_pairs(
    seed_fn=random_aa_seq, alphabet=AA_ALPHABET, n_pairs=N_HELD)

print('Building held-out 2: real CATH AA targeted perturbations…')
def cath_aa_seed(): return test_aa_seqs[int(rng.integers(0, len(test_aa_seqs)))]
held_cath_aa = make_targeted_perturbation_pairs(
    seed_fn=cath_aa_seed, alphabet=AA_ALPHABET, n_pairs=N_HELD)

print('Building held-out 3: real CATH SS targeted perturbations (cross-rep)…')
def cath_ss_seed(): return test_ss_seqs[int(rng.integers(0, len(test_ss_seqs)))]
held_cath_ss = make_targeted_perturbation_pairs(
    seed_fn=cath_ss_seed, alphabet=SS_ALPHABET, n_pairs=N_HELD)

print(f'  random AA:  {len(held_random)}')
print(f'  CATH AA:    {len(held_cath_aa)}')
print(f'  CATH SS:    {len(held_cath_ss)}')

## 8. Predict helper + banded eval metrics

Six numbers per held-out set:

1. **Aggregate Pearson r** (over [0.28, 1.0]) — for continuity with colab13.
2. **Per-band Pearson r** — restricted to true `high` and to true `mid`.
3. **AUROC for is-high-sim** — binary task: positive iff true `normLev ≥ 0.70`.
4. **High-sim band MAE** — calibration check on the band we care most about.
5. **Predicted-band confusion matrix** — 3×3, predicted band vs true band.
6. **Top-k retrieval precision** — computed in the next cell.

In [ ]:
def predict_pairs(pairs, batch_size=512):
    model.eval()
    true_v, pred_v = [], []
    with torch.no_grad():
        for i in range(0, len(pairs), batch_size):
            batch = pairs[i:i+batch_size]
            a = torch.stack([encode_pad(p[0]) for p in batch]).to(device)
            b = torch.stack([encode_pad(p[1]) for p in batch]).to(device)
            pred = model(a, b).cpu().numpy()
            true_v.extend([p[2] for p in batch])
            pred_v.extend(pred.tolist())
    return np.array(true_v), np.array(pred_v)

def banded_metrics(name, true_v, pred_v):
    if len(true_v) < 10:
        print(f'{name}: n={len(true_v)} too few'); return None
    # Aggregate
    pr_all, _ = pearsonr(true_v, pred_v)
    sr_all, _ = spearmanr(true_v, pred_v)
    # Per-band r
    hi = true_v >= BAND_HIGH
    mi = (true_v >= BAND_LOW) & (true_v < BAND_HIGH)
    pr_hi = pearsonr(true_v[hi], pred_v[hi])[0] if hi.sum() > 10 else float('nan')
    pr_mi = pearsonr(true_v[mi], pred_v[mi])[0] if mi.sum() > 10 else float('nan')
    # AUROC for is-high
    y_hi = (true_v >= BAND_HIGH).astype(int)
    auroc = roc_auc_score(y_hi, pred_v) if 0 < y_hi.sum() < len(y_hi) else float('nan')
    # MAE on high band
    mae_hi = np.abs(pred_v[hi] - true_v[hi]).mean() if hi.sum() > 0 else float('nan')
    # Confusion (band vs band)
    tb = band_labels_arr(true_v)
    pb = band_labels_arr(pred_v)
    cm = confusion_matrix(tb, pb, labels=['far','mid','high'])

    print(f'--- {name} (n={len(true_v)}) ---')
    print(f'  Pearson r (all):     {pr_all:+.3f}    Spearman: {sr_all:+.3f}')
    print(f'  Pearson r (high≥{BAND_HIGH}):  {pr_hi:+.3f}  [n_high={hi.sum()}]')
    print(f'  Pearson r (mid):     {pr_mi:+.3f}  [n_mid={mi.sum()}]')
    print(f'  AUROC (is-high):     {auroc:.3f}')
    print(f'  MAE on high band:    {mae_hi:.3f}')
    print(f'  Confusion (rows=true, cols=pred) [far, mid, high]:')
    for i, b in enumerate(['far','mid','high']):
        row = cm[i]
        denom = max(1, row.sum())
        pct = ' '.join(f'{row[j]/denom*100:5.1f}%' for j in range(3))
        print(f'    true {b:>4}: {row}   ({pct})')
    return {'pr_all': pr_all, 'pr_hi': pr_hi, 'pr_mi': pr_mi,
            'auroc': auroc, 'mae_hi': mae_hi, 'cm': cm}

true_r, pred_r = predict_pairs(held_random)
true_a, pred_a = predict_pairs(held_cath_aa)
true_s, pred_s = predict_pairs(held_cath_ss)

m_r = banded_metrics('1. random AA (in-domain)',   true_r, pred_r)
m_a = banded_metrics('2. CATH AA (dist. shift)',   true_a, pred_a)
m_s = banded_metrics('3. CATH SS (cross-rep)',     true_s, pred_s)

## 9. Visualization — confusion matrices, PR curves, band histograms

In [ ]:
def plot_confusion(cm, ax, title):
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2])
    ax.set_xticklabels(['far','mid','high']); ax.set_yticklabels(['far','mid','high'])
    ax.set_xlabel('Predicted band'); ax.set_ylabel('True band'); ax.set_title(title)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{cm[i,j]}\n({cm_norm[i,j]*100:.0f}%)',
                    ha='center', va='center',
                    color='white' if cm_norm[i,j] > 0.5 else 'black', fontsize=9)
    return im

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, m, title in zip(axes,
                        [m_r, m_a, m_s],
                        ['Random AA', 'CATH AA', 'CATH SS (cross-rep)']):
    plot_confusion(m['cm'], ax, f'{title}\nAUROC(high)={m["auroc"]:.3f}')
plt.tight_layout(); plt.show()

In [ ]:
# PR curves for binary "is high-sim"
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (true_v, pred_v, title) in zip(axes, [
    (true_r, pred_r, 'Random AA'),
    (true_a, pred_a, 'CATH AA'),
    (true_s, pred_s, 'CATH SS (cross-rep)'),
]):
    y = (true_v >= BAND_HIGH).astype(int)
    if 0 < y.sum() < len(y):
        prec, rec, _ = precision_recall_curve(y, pred_v)
        auroc = roc_auc_score(y, pred_v)
        ax.plot(rec, prec)
        ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
        ax.set_title(f'{title}\nis-high-sim  AUROC={auroc:.3f}  base rate={y.mean():.2f}')
        ax.grid(True, alpha=0.3); ax.set_xlim(0,1); ax.set_ylim(0,1.05)
    else:
        ax.set_title(f'{title}\n(degenerate label split)')
plt.tight_layout(); plt.show()

In [ ]:
# Prediction histograms per true band
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {'far': '#888', 'mid': '#3a7', 'high': '#d33'}
for ax, (true_v, pred_v, title) in zip(axes, [
    (true_r, pred_r, 'Random AA'),
    (true_a, pred_a, 'CATH AA'),
    (true_s, pred_s, 'CATH SS (cross-rep)'),
]):
    tb = band_labels_arr(true_v)
    for b in ('far','mid','high'):
        m = (tb == b)
        if m.sum() > 0:
            ax.hist(pred_v[m], bins=30, alpha=0.55, density=True,
                    label=f'true {b} (n={m.sum()})', color=colors[b])
    ax.axvline(BAND_LOW, color='k', ls=':', lw=0.8)
    ax.axvline(BAND_HIGH, color='k', ls=':', lw=0.8)
    ax.set_xlabel('Predicted normLev'); ax.set_ylabel('Density')
    ax.set_title(f'{title} — prediction distribution per true band')
    ax.set_xlim(-0.05, 1.05); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 10. High-sim sharpness diagnostics

**Identity pair test:** for input pair `(X, X)`, the architecture *should* output 1.0 (the encoder produces identical embeddings → distance 0 → similarity 1). This isn't enforced by the loss directly, so it's worth confirming.

**Saturation/compression check:** scatter restricted to true normLev `≥ 0.70` — is the model still resolving differences inside the high band, or does it saturate at one value?

In [ ]:
# Identity pair test
identity_proteins = test_aa_seqs[:300]
identity_pairs = [(p, p, 1.0) for p in identity_proteins]
tv_id, pv_id = predict_pairs(identity_pairs)
print('Identity pair (X, X) predictions on 300 test30 proteins:')
print(f'  mean pred = {pv_id.mean():.4f}  (ideal: 1.0)')
print(f'  std pred  = {pv_id.std():.4f}')
print(f'  min pred  = {pv_id.min():.4f}')

plt.figure(figsize=(8, 3))
plt.hist(pv_id, bins=30, edgecolor='k')
plt.axvline(1.0, color='r', ls='--', label='ideal = 1.0')
plt.xlabel('Predicted normLev'); plt.ylabel('Count')
plt.title('Identity-pair predictions')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# High-sim region scatter (true >= 0.7)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (true_v, pred_v, title) in zip(axes, [
    (true_r, pred_r, 'Random AA'),
    (true_a, pred_a, 'CATH AA'),
    (true_s, pred_s, 'CATH SS (cross-rep)'),
]):
    m = true_v >= BAND_HIGH
    if m.sum() < 10:
        ax.set_title(f'{title} — too few high-sim'); continue
    ax.scatter(true_v[m], pred_v[m], alpha=0.25, s=8)
    ax.plot([BAND_HIGH, 1], [BAND_HIGH, 1], 'r-', lw=2, label='y = x')
    pr = pearsonr(true_v[m], pred_v[m])[0]
    mae = np.abs(pred_v[m] - true_v[m]).mean()
    ax.set_title(f'{title} — true ≥ 0.7\nn={m.sum()}  r={pr:+.3f}  MAE={mae:.3f}')
    ax.set_xlabel('True normLev'); ax.set_ylabel('Predicted')
    ax.set_xlim(BAND_HIGH-0.02, 1.02); ax.set_ylim(BAND_HIGH-0.05, 1.05)
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 11. Top-k retrieval

For each of `TOPK_N_QUERIES` (50) randomly chosen test30 proteins:

1. Pick `TOPK_N_CANDIDATES` (500) random other test30 proteins as a candidate pool.
2. Compute the **true** `normLev` between query and each candidate (rapidfuzz, fast).
3. Compute the **model-predicted** similarity between query and each candidate.
4. For each `k ∈ {1, 5, 10}`, report `precision@k = |true_top_k ∩ pred_top_k| / k`, averaged across queries.

Run separately on the AA representation and the SS representation of the same test30 proteins (cross-rep retrieval test).

In [ ]:
def topk_retrieval(query_seqs, candidate_seqs, k_values=TOPK_VALUES,
                   n_queries=TOPK_N_QUERIES, n_candidates=TOPK_N_CANDIDATES,
                   label='AA'):
    model.eval()
    rng_local = np.random.default_rng(SEED + hash(label) % 10000)
    q_idx = rng_local.choice(len(query_seqs), size=min(n_queries, len(query_seqs)), replace=False)

    precisions = {k: [] for k in k_values}
    overlap_counts = {k: [] for k in k_values}

    for qi in q_idx:
        q = query_seqs[qi]
        cand_pool = [i for i in range(len(candidate_seqs)) if i != qi]
        c_idx = rng_local.choice(cand_pool,
                                 size=min(n_candidates, len(cand_pool)),
                                 replace=False)
        cands = [candidate_seqs[i] for i in c_idx]

        # True normLev
        true_sims = np.array([fast_norm_lev(q, c) for c in cands])

        # Predicted similarity
        with torch.no_grad():
            a = encode_pad(q).unsqueeze(0).repeat(len(cands), 1).to(device)
            b = torch.stack([encode_pad(c) for c in cands]).to(device)
            pred_sims = model(a, b).cpu().numpy()

        for k in k_values:
            true_topk = set(np.argsort(-true_sims)[:k])
            pred_topk = set(np.argsort(-pred_sims)[:k])
            overlap = len(true_topk & pred_topk)
            precisions[k].append(overlap / k)
            overlap_counts[k].append(overlap)

    print(f'--- Top-k retrieval ({label}) ---')
    print(f'  Queries: {len(q_idx)}   Candidates per query: {n_candidates}')
    for k in k_values:
        mean_p = np.mean(precisions[k])
        # Random baseline: k/n_candidates
        baseline = k / n_candidates
        print(f'  precision@{k:>2}: {mean_p:.3f}    (random baseline {baseline:.4f})')
    return precisions

print('AA representation retrieval:')
topk_aa = topk_retrieval(test_aa_seqs, test_aa_seqs, label='AA')
print()
print('SS representation retrieval (cross-rep):')
topk_ss = topk_retrieval(test_ss_seqs, test_ss_seqs, label='SS')

In [ ]:
# Plot top-k retrieval
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (topk_dict, label) in zip(axes, [(topk_aa, 'CATH AA'), (topk_ss, 'CATH SS')]):
    ks = list(topk_dict.keys())
    means = [np.mean(topk_dict[k]) for k in ks]
    stds  = [np.std(topk_dict[k]) for k in ks]
    baselines = [k / TOPK_N_CANDIDATES for k in ks]
    x = np.arange(len(ks))
    ax.bar(x - 0.2, means, 0.4, yerr=stds, label='Model', color='#3a7', capsize=4)
    ax.bar(x + 0.2, baselines, 0.4, label='Random baseline', color='#888')
    ax.set_xticks(x); ax.set_xticklabels([f'p@{k}' for k in ks])
    ax.set_ylabel('Precision'); ax.set_title(f'Top-k retrieval — {label}')
    ax.set_ylim(0, 1.05); ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## 12. Embedding space — distance histograms, PCA, t-SNE

Three views of the geometry:

1. **Embedding distance histogram per true band** — for each held-out pair, compute `‖ea − eb‖₂`; plot one histogram per true band. Clean separation between high/mid/far histograms = the embedding space has the structure we want.

2. **PCA of pair-difference vectors** — for each held-out pair, project `|ea − eb|` (element-wise absolute difference, swap-symmetric, 128-dim) to 2D via PCA; color by true band. Honest about distances (linear projection).

3. **t-SNE of per-protein embeddings** — encode every test30 protein, project to 2D, color by SuperFamily (restricted to the most populated superfamilies for visual clarity). Answers: does the encoder learn biologically meaningful structure?

In [ ]:
def get_pair_embeddings(pairs, batch_size=256):
    """Return (ea, eb) tensors for a list of pairs."""
    model.eval()
    eas, ebs = [], []
    with torch.no_grad():
        for i in range(0, len(pairs), batch_size):
            batch = pairs[i:i+batch_size]
            a = torch.stack([encode_pad(p[0]) for p in batch]).to(device)
            b = torch.stack([encode_pad(p[1]) for p in batch]).to(device)
            eas.append(model.encode(a).cpu().numpy())
            ebs.append(model.encode(b).cpu().numpy())
    return np.concatenate(eas, 0), np.concatenate(ebs, 0)

# Embedding-distance histogram per true band
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (pairs, name) in zip(axes, [
    (held_random, 'Random AA'),
    (held_cath_aa, 'CATH AA'),
    (held_cath_ss, 'CATH SS (cross-rep)'),
]):
    ea, eb = get_pair_embeddings(pairs)
    dist = np.linalg.norm(ea - eb, axis=1)
    true_v = np.array([p[2] for p in pairs])
    tb = band_labels_arr(true_v)
    for b in ('far','mid','high'):
        m = (tb == b)
        if m.sum() > 0:
            ax.hist(dist[m], bins=30, alpha=0.55, density=True,
                    label=f'true {b} (n={m.sum()})', color=colors[b])
    ax.set_xlabel('Embedding distance ‖ea − eb‖₂')
    ax.set_ylabel('Density'); ax.set_title(f'{name}')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# PCA of pair-difference vectors (use CATH AA held-out)
ea_a, eb_a = get_pair_embeddings(held_cath_aa)
diff_a = np.abs(ea_a - eb_a)
true_a_arr = np.array([p[2] for p in held_cath_aa])
tb_a = band_labels_arr(true_a_arr)

pca = PCA(n_components=2, random_state=SEED)
proj = pca.fit_transform(diff_a)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# Color by true band
for ax, (proj_, tb_, true_, name) in zip(axes, [
    (proj, tb_a, true_a_arr, 'CATH AA pair-difference PCA — colored by band'),
    (proj, tb_a, true_a_arr, 'CATH AA pair-difference PCA — colored by true normLev'),
]):
    if 'band' in name:
        for b in ('far','mid','high'):
            m = (tb_ == b)
            if m.sum() > 0:
                ax.scatter(proj_[m,0], proj_[m,1], alpha=0.4, s=10,
                           label=f'{b} (n={m.sum()})', color=colors[b])
        ax.legend()
    else:
        sc = ax.scatter(proj_[:,0], proj_[:,1], c=true_, cmap='viridis',
                        alpha=0.5, s=10, vmin=0, vmax=1)
        plt.colorbar(sc, ax=ax, label='true normLev')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
    ax.set_title(name); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'PCA explained variance (first 2): {pca.explained_variance_ratio_[:2].sum()*100:.1f}%')

In [ ]:
# t-SNE of per-protein embeddings, colored by SuperFamily
print('Encoding all test30 proteins…')
model.eval()
all_embeds = []
batch = 256
with torch.no_grad():
    for i in range(0, len(test_aa_seqs), batch):
        batch_seqs = test_aa_seqs[i:i+batch]
        x = torch.stack([encode_pad(s) for s in batch_seqs]).to(device)
        all_embeds.append(model.encode(x).cpu().numpy())
all_embeds = np.concatenate(all_embeds, 0)
print(f'  {all_embeds.shape[0]} proteins, embedding dim {all_embeds.shape[1]}')

sf = test_df['SuperFamily'].astype(str).values
sf_counts = pd.Series(sf).value_counts()
top_sfs = sf_counts.head(10).index.tolist()
mask = np.isin(sf, top_sfs)
print(f'  showing top-10 superfamilies (covers {mask.sum()} / {len(sf)} proteins)')

print('Running t-SNE…')
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, init='pca', learning_rate='auto')
proj_tsne = tsne.fit_transform(all_embeds)

plt.figure(figsize=(10, 8))
plt.scatter(proj_tsne[~mask, 0], proj_tsne[~mask, 1],
            c='#ddd', s=6, alpha=0.4, label='other')
palette = plt.cm.tab10(np.linspace(0, 1, len(top_sfs)))
for i, sf_id in enumerate(top_sfs):
    m = sf == sf_id
    plt.scatter(proj_tsne[m, 0], proj_tsne[m, 1],
                c=[palette[i]], s=20, alpha=0.8,
                label=f'{sf_id} (n={m.sum()})')
plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2')
plt.title('t-SNE of per-protein embeddings — top-10 SuperFamilies')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 13. Natural-pair eval — band-stratified

Natural CATH pairs (from `cath_s20_pairs_sample.csv.gz` and `cath_s20_pairs_high.csv.gz`) are mostly in the `far` band by the new thresholds — that's *fine*, the model is allowed to default on them. What we want to check: does the model still correctly assign these pairs to the `far` band? And on `pairs_high` (which extends into the `mid` and a sliver of `high` bands), does the model resolve them correctly?

In [ ]:
PAIR_COLS = ['domain_p2', 'domain_p1', 'ss_score', 'aa_score', 'TM_min', 'TM_max', 'cath_superFamily']
id_to_aa = dict(zip(test_df['domain_id'].astype(str), test_df['aa_seq']))
test_id_set = set(id_to_aa.keys())

def load_natural_pairs(path):
    df = pd.read_csv(path, header=None, names=PAIR_COLS, compression='gzip')
    df = df[df['domain_p1'].isin(test_id_set) & df['domain_p2'].isin(test_id_set)].copy()
    pairs = []
    for _, row in df.iterrows():
        a = id_to_aa[row['domain_p1']]; b = id_to_aa[row['domain_p2']]
        pairs.append((a, b, float(row['aa_score'])))
    return pairs

print('Loading natural pairs (random sample)…')
nat_random = load_natural_pairs(os.path.join(DATA_DIR, 'cath_s20_pairs_sample.csv.gz'))
print('Loading natural pairs (high-sim file)…')
nat_high = load_natural_pairs(os.path.join(DATA_DIR, 'cath_s20_pairs_high.csv.gz'))
print(f'  random: {len(nat_random)}, high: {len(nat_high)}')

true_n, pred_n = predict_pairs(nat_random)
true_h, pred_h = predict_pairs(nat_high)

m_n = banded_metrics('4a. CATH natural (random)', true_n, pred_n)
m_h = banded_metrics('4b. CATH natural (high)',   true_h, pred_h)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (tv, pv, title) in zip(axes, [
    (true_n, pred_n, '4a. CATH natural — random sample'),
    (true_h, pred_h, '4b. CATH natural — high-sim subset'),
]):
    ax.scatter(tv, pv, alpha=0.20, s=8)
    ax.plot([0, 1], [0, 1], 'r-', lw=2, label='y = x')
    ax.axhline(BAND_LOW, color='k', ls=':', lw=0.5)
    ax.axhline(BAND_HIGH, color='k', ls=':', lw=0.5)
    pr = pearsonr(tv, pv)[0] if len(tv) > 10 else float('nan')
    ax.set_title(f'{title}\nn={len(tv)}  r={pr:+.3f}')
    ax.set_xlabel('True aa_score (normLev)'); ax.set_ylabel('Predicted')
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 14. Comparison table — colab13 vs colab14

Fill in after running. Headline numbers shifted from "aggregate Pearson r" to **AUROC for is-high-sim** and **top-k retrieval precision**, with per-band r as the supporting detail.

| metric | colab13 | colab14 | reading |
|---|---|---|---|
| Pearson r — in-domain (all)         | +0.713 | ? | aggregate continuity |
| Pearson r — CATH AA (all)           | +0.708 | ? | aggregate continuity |
| Pearson r — CATH SS (all)           | +0.789 | ? | aggregate continuity |
| **Pearson r — CATH AA (high band)** | n/a    | ? | new headline: high-band resolution |
| **AUROC is-high-sim — CATH AA**     | n/a    | ? | new headline: detection quality |
| **AUROC is-high-sim — CATH SS**     | n/a    | ? | cross-rep detection |
| **top-1 precision — CATH AA**       | n/a    | ? | biologically resonant |
| **top-5 precision — CATH AA**       | n/a    | ? | as above |
| **top-1 precision — CATH SS**       | n/a    | ? | cross-rep retrieval |
| Identity-pair mean prediction       | n/a    | ? | calibration sanity (ideal 1.0) |
| MAE on high band — CATH AA          | n/a    | ? | calibration in the band we care about |

## What success looks like

- **AUROC ≥ 0.85** on is-high-sim for CATH AA and **≥ 0.75** for CATH SS — clean band detection, with cross-rep degradation.
- **top-1 precision ≥ 0.40** on CATH AA — the model's #1 predicted neighbor is the true #1 in 40%+ of queries.
- **Identity prediction ≥ 0.97** — high-end calibration intact.
- **PCA of pair-difference vectors:** visible separation between band clusters.
- **t-SNE:** at least 2–3 of the top-10 superfamilies show some clustering (suggests the encoder learned biologically meaningful structure).

## What failure looks like

- **AUROC near 0.5** — the band-weighting was too aggressive and destabilized training (try `W_HIGH=2, W_MID=1.5, W_FAR=0.8` as a softer alternative).
- **Identity prediction < 0.9** — high-band calibration broke. Likely loss instability.
- **top-k precision near random baseline** — model isn't using its high-sim signal for retrieval.
- **Band histograms overlap heavily** — the encoder isn't separating bands cleanly even though aggregate r is fine.

## If colab14 succeeds

The thesis pivots cleanly: instead of "approximate normLev across [0,1]", the claim becomes "**detect biologically meaningful similarity bands** — including cross-representation transfer to SS — with a single trained encoder." That story has a defensible band, a headline AUROC, top-k retrieval as the biologically resonant metric, and the PCA/t-SNE/identity diagnostics as supporting evidence.